# 7.14 少量錯誤答案會被學走嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**故意給1+1=3，loss有辦法知道這是錯的嗎？

錯誤標籤也會產生方向一致的梯度；模型不會自動知道人寫的答案錯了。低train loss可能表示把錯誤也背會了。

**把直覺寫成數學：**CE只比較提供的target，不包含外部真實世界驗證。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
logits = torch.tensor([[0.0, 0.0, 3.0, 1.0]])
for label in (2, 3):
    print("提供的label", label, "CE", F.cross_entropy(logits, torch.tensor([label])).item())

**如何讀結果：**要用獨立正確的held-out資料檢查；對照需固定樣本數只換標籤。

**只改一件事：**只提高錯誤類別分數，看錯標籤loss降低。
